# 实验八 · 递归归并排序 —— task 构造与任务并行

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐⭐⭐ 难点　|　**预计时长**：30–40 分钟

前七个实验的并行对象均为**循环**：迭代空间在进入时已知，工作量可以一次划分完毕。许多算法并不具备这一特征——递归分治、链表遍历、树与图搜索等，其工作量需在运行时逐步展开。本实验以递归归并排序为载体，引入 OpenMP 处理此类问题的构造：`task`。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. `task` 的默认数据环境为 **firstprivate**，而非并行区域中的 shared。这是使用 `task` 时需特别注意的一点，第 2.3 节将专门讨论。
> 6. cutoff（递归深度阈值）对性能影响很大。命令行第三个参数传 0 表示自动取值，第 11 节会做参数扫描。

## 🎯 学习目标

完成本实验后，学生应能够：

- 说明**规则并行**与**不规则并行**的区别，指出为何递归算法无法用 `#pragma omp for` 表达
- 掌握 `#pragma omp task` 的语义：创建一个可被任意线程执行的任务，由运行时负责调度
- 理解「根调用只能由一个线程执行」这一要求的必要性，说明去掉 `single` 会导致什么后果
- 掌握**任务调度点**的概念，说明为何生成任务的线程与等待栅栏的线程都会参与任务的执行
- 掌握 `#pragma omp taskwait` 的作用范围，并区分它与 `#pragma omp taskgroup` 的差别
- 明确 `task` 构造内变量的默认属性是 **firstprivate**，并说明这一设计的原因
- 理解 **cutoff** 的作用：任务粒度过细时，创建任务的代价会超过任务本身
- 认识**任务窃取**（work stealing）调度，理解它为何天然适合工作量不均的场景

## 🗺️ 学习路径

1. **准备阶段**：分析归并排序的递归结构与任务树形态
2. **V0 · 串行递归归并排序**：基准
3. **V1 · `task` + `taskwait`**：把两个递归调用变为两个任务，逐元素写回
4. **V2 · `task` + `memcpy`**：仅改写回方式，度量访存优化的收益
5. **可视化与分析**：讨论任务粒度、cutoff 与任务窃取
6. **扩展实验**：cutoff 扫描，寻找粒度的合适区间

## 1. 背景知识：规则并行与不规则并行

### 1.1 两类并行的区别

| | 规则并行 | 不规则并行 |
|---|---|---|
| 典型形式 | `for` 循环 | 递归、链表遍历、树与图搜索 |
| 工作量 | 进入时即可确定 | 运行时逐步展开 |
| 划分时机 | 一次划分 | 持续产生 |
| OpenMP 构造 | `#pragma omp for` | `#pragma omp task` |
| 本章实验 | 实验二至七 | **实验八** |

考虑一个链表遍历：

```c
for (node *p = head; p != NULL; p = p->next) {
  process(p);
}
```

这个循环无法用 `#pragma omp for` 分担，因为它不满足实验二 3.6 节所述的**规范形式**：循环次数在进入循环前无法确定。

递归算法面临同样的问题：递归树的形状要到运行时才展开，无法事先划分。

### 1.2 归并排序的递归结构

```text
                   [0 .. 15]
                  /         \
            [0..7]           [8..15]
           /      \          /      \
       [0..3]  [4..7]   [8..11]  [12..15]
        /  \     /  \     /  \      /  \
      ...  ...  ... ...  ... ...   ... ...
```

**关键性质**：同一层的两个子问题彼此**完全独立**，可以并行处理；而父节点的归并操作必须在两个子节点均完成之后才能开始。

这正是 `task` 与 `taskwait` 所要表达的关系：

- `task`：把左右两半各自登记为一个可独立执行的任务；
- `taskwait`：等待两个子任务完成，然后执行归并。

### 1.3 为何不能用嵌套的 `parallel for`

理论上可在每一层递归中创建并行区域，但这将导致**嵌套并行**：深度为 $d$ 的递归会产生 $2^d$ 个线程组。以 $d = 20$ 计，线程组数量将达百万量级，远超一般系统的可承受范围。

自 OpenMP 5.0 起，`max-active-levels-var` 的默认值规定为 1，即默认不启用嵌套并行；在更早的规范中，该行为由现已弃用的 `OMP_NESTED` 控制，默认同样关闭。因此内层的 `parallel` 会退化为单线程执行，并行效果仅存在于最外层。

`task` 的设计旨在解决这一问题：**任务的数量可以远多于线程数**，由运行时将任务分配给固定数量的线程执行。

## 2. 任务并行的基本构造

### 2.1 `task` 构造

```c
#pragma omp task
{ /* 任务体 */ }
```

遇到 `task` 时，当前线程将该结构化块**封装**为一个任务并放入任务池，随后**继续执行**后续代码，并不等待该任务完成。任务池中的任务由线程组中的任意空闲线程取出执行。

| 概念 | 含义 |
|---|---|
| 任务生成 | 由遇到 `task` 制导语句的线程负责封装 |
| 任务执行 | 线程组中任意线程都可能执行 |
| 执行时机 | 由运行时决定，不保证立即执行 |
| 数据环境 | 打包时**捕获**，见 2.3 节 |

### 2.2 由单个线程执行根调用

```c
#pragma omp parallel num_threads(p)
{
#pragma omp single
  {
    mergesort_task(src, dst, 0, n - 1, cutoff);   // 只由一个线程调用
  }
}
```

该结构可能引起疑问：既然仅由一个线程进入 `single` 块，为何仍需创建 $p$ 个线程？

**原因在于：`single` 限制的是**根调用**的执行次数，而非任务的生成主体或执行主体。**

```text
  single 块内的线程 0：执行根调用，生成最初的若干任务
                              │
                              ▼
                    ┌──── 任 务 池 ────┐
                    │   T1  T2  T3 …   │
                    └──────────────────┘
                      ▲              │
           生成新任务  │              │  取出执行
                      └──────────────┘
                    任意线程（含线程 0）
```

理解这一结构的关键概念是**任务调度点**（task scheduling point）：OpenMP 规定，线程走到任务调度点时，可以挂起当前任务、转而执行任务池中的其他任务。由此得到两条容易被忽略的性质：

**其一，其余线程并非只是「在等」。** 它们跳过 `single` 块后停在其末尾的隐式栅栏上，而**栅栏就是任务调度点**，因此它们在等待期间照样从池中取任务执行。

**其二，线程 0 也不是只「生成」不「执行」。** 递归函数的每一层都有 `#pragma omp taskwait`，线程 0 在此必须等待两个子任务完成——而 `taskwait` 同样是任务调度点，它会转而执行池中的任务而非空等。此外，多数实现（含 GCC 的 libgomp）对未完成任务数设有阈值，超过后生成线程不再入队，而是**当场执行**新创建的任务，以防任务池无限膨胀。

**其三，任务树是分布式展开的。** `single` 只保证根调用发生一次；某个任务一旦被线程 3 取走执行，其任务体内部的 `#pragma omp task` 就由线程 3 生成。因此任务的生成者同样遍及全体线程。

> 练习 2 与思考题 2 会从实验和规范两个角度回到这一点。若在叶任务处统计执行者与生成者的线程编号，会看到两者在各线程之间都是大致均匀分布的。

> **若省略 `single`**：$p$ 个线程将各自完整地生成一棵任务树，而它们操作的是**同一对 `src` / `dst` 缓冲区**。多个任务会并发地对同一区间执行 `merge`，同时读写 `dst` 的同一段——这构成数据竞争，**结果通常是错误的**，且错误不可复现。性能也会因重复劳动而显著下降。练习 2 将实测这一点。
>
> 需要说明的是，「排序具有幂等性」不能用来论证此处的正确性：幂等性指的是对**已排序**数组再排一次结果不变，而这里发生的是多个线程并发改写同一块内存的中间状态，两者是不同的问题。

> **能否用 `master` 替代**：可以。其余线程跳过 `master` 块后会停在**并行区域末尾**的隐式栅栏上，而栅栏同样是任务调度点，它们在此照样参与任务执行。二者的差别仅在于等待点的位置。
>
> 本实验仍推荐 `single`，理由有二：一是它更直接地表达「由任意一个线程执行一次」这一意图，而 `master` 额外绑定了「必须是 0 号线程」这一无关约束；二是 `master` 构造在 OpenMP 5.1 中已被列为弃用特性，由 `masked` 取代（见实验五 3.2 节）。
>
> 但有一点不能变：**根调用必须由且仅由一个线程执行**。无论使用 `single` 还是 `master`，去掉它都会导致上述的数据竞争。

### 2.3 `task` 的默认数据环境是 `firstprivate`

**这是 `task` 与其他构造的重要区别，也是使用时需特别注意之处。**

| 构造 | 外层局部变量的默认属性 |
|---|---|
| `#pragma omp parallel` | **shared** |
| `#pragma omp for` | 继承外层 |
| `#pragma omp task` | **firstprivate** |

**为何如此设计**：任务的执行时机由运行时决定，可能远晚于它被创建的时刻。此时创建该任务的函数调用可能**已经返回**，其栈帧已失效。若任务以共享方式引用这些局部变量，将访问到已被回收的内存。

把标量按值捕获（`firstprivate`），任务便携带了自己所需的全部信息，与创建者的生命周期解耦。

本实验的写法显式列出了两类变量，以便阅读时对照：

```c
#pragma omp task firstprivate(left, mid, cutoff) shared(src, dst)
{ mergesort_task_v1(src, dst, left, mid, cutoff); }
```

- `left`、`mid`、`cutoff` 是**标量**，按值捕获，各任务互不干扰；
- `src`、`dst` 是**指针**，指向长期存在的堆内存，声明为 `shared`，各任务共享同一块数据。

> **注意区分「指针共享」与「数据共享」**：即便写成 `firstprivate(src)`，复制的也只是指针的值，各任务仍然访问同一块内存。此处写 `shared` 是为了明确表达意图。

### 2.4 `taskwait` 与 `taskgroup`

```c
#pragma omp taskwait     // 等待当前任务的「直接子任务」
#pragma omp taskgroup    // 等待块内产生的「全部后代任务」
{ ... }
```

| | `taskwait` | `taskgroup` |
|---|---|---|
| 等待范围 | 仅**直接**子任务 | 全部**后代**任务 |
| 形式 | 独立的制导语句 | 包裹一个结构化块 |
| 开销 | 较低 | 较高 |

**本实验用 `taskwait` 即可**。归并排序的递归中，每一层只需等待自己创建的两个直接子任务；而那两个子任务内部又各自有 `taskwait` 保证其后代已完成。这种**逐层等待**的结构使得 `taskwait` 足够，无需 `taskgroup`。

若某个算法在一处创建了大量互不嵌套的任务，并需要一次性等待全部完成，则 `taskgroup` 更为合适。

## 3. 任务粒度与调度

### 3.1 cutoff：任务粒度的控制

```c
if (right - left + 1 <= cutoff) {
  mergesort_serial(src, dst, left, right);   // 退回串行
  return;
}
```

创建一个任务需要分配任务描述符、复制 `firstprivate` 变量、入队、以及后续的出队与调度，其代价通常在**亚微秒到微秒量级**。

若不加限制地递归到单个元素，任务数量将达到 $O(n)$ 量级。以 $n = 4\times10^6$ 计，那是数百万个任务，**创建任务的总开销会远超排序本身**。

cutoff 的作用是：当子问题规模小于阈值时，不再创建任务，直接串行处理。

本实验的默认取值为：

```c
cutoff = n / (4 * thread_count);   // 下限 1000
```

其含义是让**叶任务**的数量约为线程数的 4 倍——既足以让运行时有调度余地（应对工作量不均），又不至于产生过多的任务开销。

> 需要区分两个数量：cutoff 决定的是**叶任务**数（约 $n/\text{cutoff}$）。由于递归树上每个内部节点都会创建两个任务，实际**创建的任务总数**约为叶任务数的两倍。第 11 节的扫描按叶任务数标注。

### 3.2 任务窃取调度

多数 OpenMP 实现采用**任务窃取**（work stealing）：每个线程维护自己的任务队列，当本地队列为空时，从其他线程的队列尾部「窃取」任务。

```text
  线程0 队列: [T1][T2][T3]          线程3 队列: 空
                          ↖────────────────┘
                        线程3 从线程0 尾部窃取 T3
```

**该机制天然适合工作量不均衡的场景**：无需像实验四那样预先选择调度策略，空闲线程会自动从其他线程的任务队列中获取任务。

| | `#pragma omp for` + `schedule` | `task` + 任务窃取 |
|---|---|---|
| 划分时机 | 进入循环时 | 持续产生 |
| 均衡机制 | 预先选择调度策略 | 运行时自动窃取 |
| 适用 | 迭代空间已知 | 工作量运行时才展开 |
| 开销 | 低 | 较高（每个任务都有描述符） |

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。第四章的 GEMV 实验已经遇到过由此引发的测量异常，本章的每一处性能数据同样需要在这一前提下解读。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_mergesort'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 版本设计总览

| 版本 | 新增计算函数 | 并行方式 | 写回方式 | 结果表行号 |
|---|---|---|---|---|
| **V0** | `mergesort_serial` | 串行递归 | 逐元素循环 | 1 |
| **V1** | `mergesort_task_v1` | `task` + `taskwait` | 逐元素循环 | 2 |
| **V2** | `mergesort_task_v2` | `task` + `taskwait` | `memcpy` | 3 |

**两段对照的读法**：

| 对照 | 度量的对象 |
|---|---|
| V0 → V1 | 任务并行的净收益（并行加速减去任务开销） |
| V1 → V2 | 写回方式的差异，即 `memcpy` 相对逐元素循环的收益 |

V1 与 V2 的任务结构完全相同，唯一的差别是归并结果写回源数组的方式。因此第二段对照与并行无关，它衡量的是一处纯粹的访存优化。

### 6.1 关于校验方式

三个版本各自测试结束后**立即**校验，且以 `qsort` 的结果为参照**逐元素比对**。这两点都是必要的：

- 若把校验统一放在全部测试之后，检查的就只是最后一个版本留下的数据，前两行的结论毫无意义；
- 若只检查「结果是否非降序」，则无法发现元素丢失与元素重复。逐元素比对能同时覆盖顺序错误、丢失与重复三类缺陷。

## 7. 逐版本代码讲解

### 7.1 双缓冲归并

本实验的归并采用**双缓冲**结构：`src` 存放数据，`dst` 作为暂存区。

```c
static void merge(const int *src, int *dst, int left, int mid, int right) {
  int i = left, j = mid + 1, k = left;
  while (i <= mid && j <= right) {
    dst[k++] = (src[i] <= src[j]) ? src[i++] : src[j++];
  }
  while (i <= mid)   dst[k++] = src[i++];
  while (j <= right) dst[k++] = src[j++];
}
```

归并的结果落在 `dst` 中，随后必须写回 `src`，以维持「排序结果始终留在 `src`」这一约定。V1 与 V2 的差别正在于这一步。

### 7.2 V0 · 串行递归

```c
static void mergesort_serial(int *src, int *dst, int left, int right) {
  if (left >= right) return;

  int mid = left + (right - left) / 2;
  mergesort_serial(src, dst, left, mid);
  mergesort_serial(src, dst, mid + 1, right);
  merge(src, dst, left, mid, right);

  for (int i = left; i <= right; i++) {
    src[i] = dst[i];
  }
}
```

> `int mid = left + (right - left) / 2;` 而非 `(left + right) / 2`，是为避免 `left + right` 在大规模下发生整数溢出。这是一种应当遵循的书写规范。

### 7.3 V1 · `task` + `taskwait`

```c
static void mergesort_task_v1(int *src, int *dst, int left, int right,
                              int cutoff) {
  if (right - left + 1 <= cutoff) {
    mergesort_serial(src, dst, left, right);   // 粒度足够小，退回串行
    return;
  }

  int mid = left + (right - left) / 2;

#pragma omp task firstprivate(left, mid, cutoff) shared(src, dst)
  { mergesort_task_v1(src, dst, left, mid, cutoff); }

#pragma omp task firstprivate(mid, right, cutoff) shared(src, dst)
  { mergesort_task_v1(src, dst, mid + 1, right, cutoff); }

#pragma omp taskwait          // 两个子任务都完成后才能归并

  merge(src, dst, left, mid, right);

  for (int i = left; i <= right; i++) {
    src[i] = dst[i];
  }
}
```

**以下四点需要注意**：

1. **cutoff 检查放在最前面**。规模足够小时直接调用串行版本，不再创建任务。
2. **两个 `task` 之间不阻塞**。创建第一个任务后，当前线程立即继续创建第二个，两者可以被不同线程并行执行。
3. **`taskwait` 是必需的**。若省略，`merge` 可能在子任务完成之前就开始，读到的是未排序的数据。
4. **递归调用中传递的是标量的副本**。由于 `firstprivate` 按值捕获，即使外层函数已经返回，任务体内的 `left`、`mid` 依然有效。

### 7.4 V2 · `memcpy` 写回

```c
  merge(src, dst, left, mid, right);

  memcpy(src + left, dst + left,
         (size_t)(right - left + 1) * sizeof(int));
```

任务结构与 V1 完全相同，只把逐元素的写回循环换成一次 `memcpy`。

**`memcpy` 通常更快的原因**：

- 库实现针对目标平台做了向量化与预取优化；
- 对大块数据可使用非临时存储指令，避免污染缓存；
- 无需逐次的循环判断与下标计算。

> **收益的范围有限**：cutoff 以下的子问题仍由 `mergesort_serial` 处理，而它用的是逐元素循环。因此 V2 的改进只作用于 cutoff **以上**的若干层。第 11 节的 cutoff 扫描可以观察到：cutoff 越小、递归层数越多，V2 相对 V1 的优势越明显。

### 7.5 任务的启动结构

```c
static void run_task_sort(int *src, int *dst, long n, int thread_count,
                          int cutoff, int use_memcpy) {
#pragma omp parallel num_threads(thread_count) default(none) \
    shared(src, dst, n, cutoff, use_memcpy)
  {
#pragma omp single
    {
      if (use_memcpy) {
        mergesort_task_v2(src, dst, 0, (int)n - 1, cutoff);
      } else {
        mergesort_task_v1(src, dst, 0, (int)n - 1, cutoff);
      }
    }
  }
}
```

这正是 2.2 节所述的标准形式：`parallel` 创建线程组，`single` 让其中一个线程生成任务树，其余线程在栅栏处参与任务执行。

## 8. 源代码写入

In [ ]:
%%writefile {SRC_DIR}/omp_mergesort_task.c
#define _POSIX_C_SOURCE 200809L

#include <limits.h>
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define NTIMES 3
#define MAX_THREADS 16
#define ALIGN_BYTES 64
#define MIN_CUTOFF 1000

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static void *alloc_aligned(size_t bytes) {
  size_t rounded = ((bytes + ALIGN_BYTES - 1) / ALIGN_BYTES) * ALIGN_BYTES;
  return aligned_alloc(ALIGN_BYTES, rounded);
}

static int check_equal_int(const int *ref, const int *test, long n) {
  for (long i = 0; i < n; i++) {
    if (ref[i] != test[i]) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Method", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

static int compare_ints(const void *p, const void *q) {
  int x = *(const int *)p;
  int y = *(const int *)q;
  return (x > y) - (x < y);
}

// ----------------------------------------------------------------------------
// Merge src[left..mid] and src[mid+1..right] into dst[left..right].
// ----------------------------------------------------------------------------
static void merge(const int *src, int *dst, int left, int mid, int right) {
  int i = left;
  int j = mid + 1;
  int k = left;

  while (i <= mid && j <= right) {
    if (src[i] <= src[j]) {
      dst[k++] = src[i++];
    } else {
      dst[k++] = src[j++];
    }
  }
  while (i <= mid) {
    dst[k++] = src[i++];
  }
  while (j <= right) {
    dst[k++] = src[j++];
  }
}

// ============================================================================
// V0: Serial recursive merge sort. dst is scratch space, the sorted range is
// left in src.
// ============================================================================
static void mergesort_serial(int *src, int *dst, int left, int right) {
  if (left >= right) {
    return;
  }

  int mid = left + (right - left) / 2;

  mergesort_serial(src, dst, left, mid);
  mergesort_serial(src, dst, mid + 1, right);

  merge(src, dst, left, mid, right);

  for (int i = left; i <= right; i++) {
    src[i] = dst[i];
  }
}

// ============================================================================
// V1: one task per half, element-wise write-back.
// ============================================================================
static void mergesort_task_v1(int *src, int *dst, int left, int right,
                              int cutoff) {
  if (right - left + 1 <= cutoff) {
    mergesort_serial(src, dst, left, right);
    return;
  }

  int mid = left + (right - left) / 2;

#pragma omp task firstprivate(left, mid, cutoff) shared(src, dst)
  { mergesort_task_v1(src, dst, left, mid, cutoff); }

#pragma omp task firstprivate(mid, right, cutoff) shared(src, dst)
  { mergesort_task_v1(src, dst, mid + 1, right, cutoff); }

  // Wait for both child tasks before merging their results.
#pragma omp taskwait

  merge(src, dst, left, mid, right);

  for (int i = left; i <= right; i++) {
    src[i] = dst[i];
  }
}

// ============================================================================
// V2: identical task structure, memcpy write-back.
// ============================================================================
static void mergesort_task_v2(int *src, int *dst, int left, int right,
                              int cutoff) {
  if (right - left + 1 <= cutoff) {
    mergesort_serial(src, dst, left, right);
    return;
  }

  int mid = left + (right - left) / 2;

#pragma omp task firstprivate(left, mid, cutoff) shared(src, dst)
  { mergesort_task_v2(src, dst, left, mid, cutoff); }

#pragma omp task firstprivate(mid, right, cutoff) shared(src, dst)
  { mergesort_task_v2(src, dst, mid + 1, right, cutoff); }

#pragma omp taskwait

  merge(src, dst, left, mid, right);

  memcpy(src + left, dst + left, (size_t)(right - left + 1) * sizeof(int));
}

// Only one thread of the team generates the initial task tree.
static void run_task_sort(int *src, int *dst, long n, int thread_count,
                          int cutoff, int use_memcpy) {
#pragma omp parallel num_threads(thread_count) default(none) \
    shared(src, dst, n, cutoff, use_memcpy)
  {
#pragma omp single
    {
      if (use_memcpy) {
        mergesort_task_v2(src, dst, 0, (int)n - 1, cutoff);
      } else {
        mergesort_task_v1(src, dst, 0, (int)n - 1, cutoff);
      }
    }
  }
}

int main(int argc, char *argv[]) {
  if (argc != 4) {
    printf("Usage: %s <n> <thread_count> <cutoff>\n", argv[0]);
    printf("Example: %s 4000000 4 0\n", argv[0]);
    return 1;
  }

  long n = strtol(argv[1], NULL, 10);
  int thread_count = (int)strtol(argv[2], NULL, 10);
  int cutoff = (int)strtol(argv[3], NULL, 10);

  if (n <= 1 || n > INT_MAX) {
    printf("Error: n must be between 2 and %d\n", INT_MAX);
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }
  if (cutoff <= 0) {
    cutoff = (int)(n / (4 * thread_count));
    if (cutoff < MIN_CUTOFF) {
      cutoff = MIN_CUTOFF;
    }
  }

  printf("%s\n", BANNER);
  printf(" Lab 8: Task Parallelism (Merge Sort)\n");
  printf(" n: %ld | Threads: %d | Cutoff: %d | Runs: %d\n", n, thread_count,
         cutoff, NTIMES);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  size_t bytes = (size_t)n * sizeof(int);
  int *a_orig = (int *)alloc_aligned(bytes);
  int *a_work = (int *)alloc_aligned(bytes);
  int *a_ref = (int *)alloc_aligned(bytes);
  int *tmp = (int *)alloc_aligned(bytes);

  if (a_orig == NULL || a_work == NULL || a_ref == NULL || tmp == NULL) {
    printf("Error: memory allocation failed\n");
    return 1;
  }

  srand(42);
  for (long i = 0; i < n; i++) {
    a_orig[i] = rand() % 1000000;
  }

  memcpy(a_ref, a_orig, bytes);
  qsort(a_ref, (size_t)n, sizeof(int), compare_ints);

  double start = 0.0;
  double t[3] = {0.0};
  int ok[3] = {0};

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    mergesort_serial(a_work, tmp, 0, (int)n - 1);
    t[0] += get_time_ms() - start;
  }
  ok[0] = check_equal_int(a_ref, a_work, n);

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    run_task_sort(a_work, tmp, n, thread_count, cutoff, 0);
    t[1] += get_time_ms() - start;
  }
  ok[1] = check_equal_int(a_ref, a_work, n);

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    run_task_sort(a_work, tmp, n, thread_count, cutoff, 1);
    t[2] += get_time_ms() - start;
  }
  ok[2] = check_equal_int(a_ref, a_work, n);

  for (int i = 0; i < 3; i++) {
    t[i] /= NTIMES;
  }

  print_table_header();
  print_row("V0: Serial Merge Sort", t[0], t[0], ok[0]);
  print_row("V1: Task + Element Copy", t[1], t[0], ok[1]);
  print_row("V2: Task + Memcpy", t[2], t[0], ok[2]);
  printf("%s\n", LINE);

  free(a_orig);
  free(a_work);
  free(a_ref);
  free(tmp);
  return 0;
}

## 9. 编译与运行

### 9.1 编译

In [ ]:
bin_task = compile_c('omp_mergesort_task.c', extra=())

### 9.2 运行

参数依次为：元素个数、线程数、cutoff（传 0 表示自动取值）。

取 $n = 4\times10^6$、4 线程时，自动 cutoff 为 $4\times10^6 / 16 = 250000$，叶任务约 16 个，是线程数的 4 倍；实际创建的任务总数约为其两倍。

In [ ]:
out_task = run_c(bin_task, 4000000, 4, 0,
                 env={'OMP_PROC_BIND': 'close', 'OMP_PLACES': 'cores'})
rows_task = parse_table(out_task)
print()
for name, ms, sp, chk in rows_task:
    print('  %-28s %9.3f ms  %5.2fx  %s' % (name, ms, sp, chk))

## 10. 结果可视化

In [ ]:
plot_speedup(rows_task,
             'Lab 8: Task-based Merge Sort (n = 4e6, 4 threads)')

## 11. cutoff 参数扫描

cutoff 决定了任务的粒度，是本实验最关键的调优参数：

| cutoff | 任务数量 | 任务开销 | 负载均衡 |
|---|---|---|---|
| 很小 | 很多 | **高** | 好 |
| 适中 | 适中 | 适中 | 好 |
| 很大 | 很少 | 低 | **差**（任务数少于线程数时无法用满） |

下面扫描一系列 cutoff 取值，寻找该平台上的合适区间。

In [ ]:
import matplotlib.pyplot as plt

N_FIX, NT = 4000000, 4
cutoffs = [1000, 5000, 20000, 60000, 125000, 250000, 500000, 1000000]
v1_ms, v2_ms, ntask = [], [], []
env_full = dict(os.environ)
env_full.update({'OMP_PROC_BIND': 'close', 'OMP_PLACES': 'cores'})

for c in cutoffs:
    out = subprocess.run([bin_task, str(N_FIX), str(NT), str(c)],
                         capture_output=True, text=True,
                         env=env_full).stdout
    d = {r[0]: r[1] for r in parse_table(out)}
    v1_ms.append(d.get('V1: Task + Element Copy', float('nan')))
    v2_ms.append(d.get('V2: Task + Memcpy', float('nan')))
    ntask.append(max(1, N_FIX // c))
    print('cutoff = %-8d  约 %5d 个叶任务   V1 %8.1f ms   V2 %8.1f ms'
          % (c, ntask[-1], v1_ms[-1], v2_ms[-1]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cutoffs, v1_ms, marker='o', label='V1: Task + Element Copy')
ax.plot(cutoffs, v2_ms, marker='s', label='V2: Task + Memcpy')
serial = [r[1] for r in rows_task if r[0].startswith('V0')]
if serial:
    ax.axhline(serial[0], color='#7f7f7f', linestyle='--',
               linewidth=1.2, label='V0: Serial Baseline')
ax.axvline(N_FIX / (4 * NT), color='gray', linestyle=':', linewidth=1.2)
ax.text(N_FIX / (4 * NT), max(v1_ms) * 0.95,
        ' auto = n/(4p)', fontsize=9, va='top')
ax.set_xscale('log')
ax.set_xlabel('cutoff (log scale)')
ax.set_ylabel('Time (ms)')
ax.set_title('Lab 8: Time vs. Task Cutoff (n = 4e6, 4 threads)')
ax.grid(linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()
print()
print('曲线两端上翘：左端因任务过多而开销高，右端因任务过少而无法用满线程。')

## 12. 结果分析

> 以下结论针对**趋势规律**。具体数值随平台、核心数与内存带宽而变化。

**① 任务并行使递归算法得以并行化**

这是 `task` 的核心价值所在。归并排序的递归树无法用 `#pragma omp for` 表达，而使用 `task` 仅需在两个递归调用处分别添加一条制导语句。

**② 加速比通常低于线程数**

原因有三：

| 来源 | 说明 |
|---|---|
| 归并阶段的串行部分 | 顶层的 `merge` 处理全部 $n$ 个元素，无法并行 |
| 任务创建与调度开销 | 每个任务都有描述符的分配与入队出队 |
| 访存带宽 | 归并是典型的流式访存，与实验七类似受带宽限制 |

**其中第一项是算法本身固有的**。顶层的 `merge` 需处理全部 $n$ 个元素，约占总工作量的 $1/\log_2 n$。但受并行度限制的不止顶层：自顶向下第 $k$ 层有 $2^k$ 个独立的归并，只要 $2^k < p$，该层就用不满线程组。把这些层累加，受限部分约为 $2/\log_2 n$，这构成了加速比上限的主要来源。若要进一步提升，需要采用**并行归并**算法（按中位数把两个有序段切分为多段后并行归并），那已超出本实验的范围。

**③ V2 相对 V1 的改进来自 `memcpy`，与并行无关**

两者的任务结构完全一致。改进幅度取决于 cutoff 以上的递归层数——cutoff 越小，被 `memcpy` 覆盖的层数越多，收益越明显。

**④ cutoff 曲线呈 U 形**

第 11 节的扫描结果通常呈现两端升高的 U 形曲线：

- **左端（cutoff 过小）**：任务数量可达百万量级，任务描述符的分配与调度开销可能超过排序本身；
- **右端（cutoff 过大）**：任务数量少于线程数，部分线程处于空闲状态；极端情况下 cutoff 大于 $n$ 时退化为纯串行；
- **中段**：任务数约为线程数的若干倍，既有调度余地又无过多开销。

**经验取值**：叶任务数取线程数的 $4$ 至 $16$ 倍通常是合适的起点。本实验的自动取值 $n/(4p)$ 正对应叶任务数为 $4p$。

**⑤ 任务窃取自动完成了负载均衡**

归并排序的递归树是平衡的，各子任务工作量相近，因此本实验中任务窃取的优势并不突出。

但若改为**快速排序**——其划分点由数据决定，两个子问题的规模可能相差悬殊——则任务窃取的优势会更为明显：无需像实验四那样预先选择调度策略，空闲线程会自动承担规模较大的子树。

## 13. 🔧 动手练习

**练习 1**　把线程数依次取 1、2、4、8（cutoff 传 0），绘制加速比曲线，估算顶层串行归并所占的比例，并用 Amdahl 定律与实测值对照。

**练习 2**　去掉 `#pragma omp single`（改为整个并行区域都调用递归函数），重新编译运行，**连续运行五次**，记录每次的耗时与校验结论。校验是否每次都通过？请从 `src` 与 `dst` 被所有线程共享这一事实出发，解释所观察到的现象。

**练习 3**　去掉 `#pragma omp taskwait`，重新编译运行，记录校验结论。请指出错误发生在归并的哪一步。

**练习 3+**（观察）　在 `mergesort_task_v1` 的 cutoff 分支处加一行统计，记录每个叶任务由哪个线程执行；再在创建任务处加一行统计生成者。运行后对比两组分布，验证 2.2 节所述的三条性质。提示：用 `omp_get_thread_num()` 取线程编号，并以 `#pragma omp atomic` 保护计数器的自增。

**练习 4**　把 `#pragma omp task` 的 `firstprivate(left, mid, cutoff)` 改为 `shared(left, mid, cutoff)`，重新编译运行。结果是否仍然正确？请结合 `taskwait` 的位置解释原因。在此基础上继续思考：若把 `taskwait` 移出该函数（例如挪到 `single` 块的末尾统一等待），这一改动是否还安全？

**练习 5**（进阶）　把 `taskwait` 换成 `taskgroup` 包裹两个 `task`，比较性能差异。在本实验的递归结构中，两者的语义是否等价？

**练习 6**（进阶）　用 `task` 改写快速排序，对比其在**已排序数组**（最坏划分）上的表现，观察任务窃取如何应对极不均衡的任务树。

**练习 7**（进阶）　把 `#pragma omp single` 换成 `#pragma omp master`，重新编译运行。结果是否正确？性能是否与原版相当？请据此评价「`master` 末尾没有隐式栅栏，因此其余线程不会参与任务执行」这一说法。

### 13.1 练习 1 的参考实现：线程数扩展性与串行比例估计

In [ ]:
import matplotlib.pyplot as plt

threads = [1, 2, 4, 8]
sp2 = []
for nt in threads:
    out = subprocess.run([bin_task, '4000000', str(nt), '0'],
                         capture_output=True, text=True,
                         env=env_full).stdout
    rows = parse_table(out)
    d = {r[0]: r[2] for r in rows}
    sp2.append(d.get('V2: Task + Memcpy', float('nan')))
    print('线程数 %-2d   V2 加速比 %5.2fx' % (nt, sp2[-1]))

# 由最大线程数处的加速比反推 Amdahl 串行比例 s
p_max, s_max = threads[-1], sp2[-1]
s = None
if s_max and s_max > 0 and p_max > 1:
    s = (p_max / s_max - 1.0) / (p_max - 1.0)

print()
if s is None or not (0.0 < s < 1.0):
    print('本机可用核心不足，加速比未随线程数增长，Amdahl 反推不适用。')
    print('请在多核平台上重新运行本单元格。')
else:
    print('由 p = %d、加速比 = %.2f 反推串行比例 s ≈ %.3f'
          % (p_max, s_max, s))
    print('据此推得加速比上限 1/s ≈ %.2f' % (1.0 / s))

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(threads, sp2, marker='o', label='V2 measured')
if s is not None and 0.0 < s < 1.0:
    ax.plot(threads, [1.0 / (s + (1 - s) / p) for p in threads],
            marker='x', linestyle='--',
            label='Amdahl (s = %.3f)' % s)
ax.plot(threads, threads, color='gray', linestyle=':', label='Ideal linear')
ax.set_xlabel('Thread count')
ax.set_ylabel('Speedup')
ax.set_title('Lab 8: Scalability vs. Amdahl Prediction')
ax.set_xticks(threads)
ax.grid(linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

## 14. 🤔 思考题

**思考题 1**　2.3 节说明 `task` 的默认数据环境是 `firstprivate`，理由是任务可能在创建者的栈帧失效之后才执行。本实验中 `left`、`mid` 等变量是否真的会遇到这一情况？请结合 `taskwait` 的位置作答。

**思考题 2**　`single` 块末尾有隐式栅栏，其余线程在此等待。既然在等待，它们又如何执行任务？请说明 OpenMP 规范中「任务调度点」这一概念在此处的作用，并进一步指出：进入 `single` 块的那个线程在整个过程中是否也执行了任务？在哪些位置？

**思考题 3**　本实验的任务树是平衡的。若改用快速排序且划分极不均衡（例如对已排序数组取首元素为主元），任务树会变成什么形状？`task` 与 `#pragma omp for` 哪一个更能应对？

**思考题 4**　顶层的 `merge` 需要处理全部 $n$ 个元素且无法并行。请估算它在总工作量中的占比，并据此说明为何归并排序的并行加速比存在一个与 $\log n$ 相关的上限。

**思考题 5**　V1 与 V2 的差别仅在写回方式。若把 `mergesort_serial` 中的逐元素写回也改为 `memcpy`，V0、V1、V2 三行的耗时会各自如何变化？这样改动之后，V1 与 V2 的对照还有意义吗？

**思考题 6**（综合）　对比本实验的 `task` 与实验四的 `schedule(dynamic)`：两者都是运行时动态分配工作。请指出它们在适用范围、开销与均衡能力上的差异，并各举一个更适合使用对方的场景。

## 15. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| 不规则并行 | 递归、链表、树与图，工作量运行时才展开 |
| `task` | 打包结构化块为任务，由任意线程执行 |
| 根调用 | 必须由且仅由一个线程执行；否则多线程重复展开会造成数据竞争 |
| 任务调度点 | `taskwait`、栅栏、任务生成处等；线程在此可转去执行池中的任务 |
| 生成与执行 | 均不限于某个线程：任务体内部还会继续生成任务，生成者与执行者遍及全体线程 |
| `single` 与 `master` | 二者在此均可用；`single` 更贴合意图，且 `master` 在 OpenMP 5.1 中已弃用 |
| 默认数据环境 | `task` 内为 **firstprivate**，与 `parallel` 的 shared 不同 |
| `taskwait` | 等待直接子任务 |
| `taskgroup` | 等待全部后代任务 |
| cutoff | 控制任务粒度，取值使任务数约为线程数的 4–16 倍 |
| 任务窃取 | 空闲线程自动从他人队列取任务，天然适应不均衡 |

### 后续内容

本章的最后是**通用矩阵乘法（GEMM）大实验**。它将把本章的全部知识点——数据环境、循环变换、调度、缓存友好的数据布局、SIMD 协同——综合运用于一个计算受限的内核，并与实验七的访存受限内核形成完整对照。